# HW03 — Model Serving & Deployment

Your model is trained and tracked in MLflow. A model that only runs in a notebook is not useful in production.

In this homework you will:

- verify that your trained model is reproducible and ready for serving
- wrap it in a FastAPI service with proper input validation and batch support
- measure the performance difference between single and batch prediction
- package the service in Docker with a size-optimized image
- write Kubernetes manifests to deploy the service

## Submission discipline

This is individual work.

Work locally. Push to GitHub. Do not SSH into the server.

Do not commit `.env`, `.venv/`, passwords, tokens, or notebook checkpoints.
Do not hardcode passwords anywhere in your code.

## Useful references

- MLflow model loading: https://mlflow.org/docs/latest/python_api/mlflow.sklearn.html
- FastAPI: https://fastapi.tiangolo.com/
- FastAPI lifespan: https://fastapi.tiangolo.com/advanced/events/
- Pydantic v2: https://docs.pydantic.dev/latest/
- Dockerfile reference: https://docs.docker.com/reference/dockerfile/
- Docker multi-stage builds: https://docs.docker.com/build/building/multi-stage/
- Kubernetes Deployments: https://kubernetes.io/docs/concepts/workloads/controllers/deployment/
- Kubernetes Services: https://kubernetes.io/docs/concepts/services-networking/service/

## What to avoid

- Loading the model inside the request handler. Load once at startup.
- Hardcoded passwords in any source file.
- A Docker image that bakes in the model file. Pull from MLflow at startup.
- Returning raw numpy types from the API. JSON needs native Python types.
- Skipping the batch vs single benchmark. The numbers tell a story.

In [1]:
import os
import re
import subprocess
import sys
import time
import textwrap
from pathlib import Path

import numpy as np
import pandas as pd
import mlflow
import mlflow.sklearn
from dotenv import load_dotenv
from mlflow.tracking import MlflowClient

def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / 'pyproject.toml').exists() and (candidate / 'src' / 'airbnb_serving').exists():
            return candidate
    raise FileNotFoundError(
        'Could not find HW03 project root. Open the notebook from HW03/HW03_model_serving/.'
    )

PROJECT = find_project_root(Path.cwd())
os.chdir(PROJECT)
load_dotenv(PROJECT / '.env')

src_path = str(PROJECT / 'src')
if src_path not in sys.path:
    sys.path.insert(0, src_path)

subprocess.check_call(
    [sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', '-e', '.'],
    cwd=PROJECT,
)

for path in ['src/airbnb_serving', 'k8s', 'reports', 'screenshots']:
    (PROJECT / path).mkdir(parents=True, exist_ok=True)
(PROJECT / 'src/airbnb_serving/__init__.py').write_text('__version__ = "0.1.0"\n')

STUDENT_ID = os.getenv('QBC12_STUDENT_ID', '') or input('GitHub username / student id: ').strip()
safe_student = re.sub(r'[^a-zA-Z0-9_]', '_', STUDENT_ID.lower())
EXPERIMENT_NAME = os.getenv('EXPERIMENT_NAME', '') or f'qbc12_hw02_{safe_student}'

print('PROJECT:', PROJECT)
print('Kernel Python:', sys.executable)
print('EXPERIMENT_NAME:', EXPERIMENT_NAME)


PROJECT: /home/samin/Desktop/mlops/MLOps-Course/HW03/HW03_model_serving
Kernel Python: /bin/python3
EXPERIMENT_NAME: qbc12_hw02_student_samin_kakaei


---
## Part 1 — Model Reproducibility Check

Before serving a model, you must verify it produces exactly the same output as it did during training.

This is called a **reproducibility check** and it catches silent bugs like:
- preprocessing mismatch between training and serving
- wrong model version loaded
- feature column order changed

### 1.1 Connect to MLflow and load your best model

In [2]:
MLFLOW_TRACKING_URI = os.getenv('MLFLOW_TRACKING_URI', 'http://185.50.38.163:33014')
MLFLOW_USERNAME = (
    os.getenv('MLFLOW_TRACKING_USERNAME', '')
    or os.getenv('MLFLOW_USERNAME', '')
    or input('MLflow username: ').strip()
)
MLFLOW_PASSWORD = (
    os.getenv('MLFLOW_TRACKING_PASSWORD', '')
    or os.getenv('MLFLOW_PASSWORD', '')
    or input('MLflow password: ').strip()
)

os.environ['MLFLOW_TRACKING_USERNAME'] = MLFLOW_USERNAME
os.environ['MLFLOW_TRACKING_PASSWORD'] = MLFLOW_PASSWORD

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
client = MlflowClient()

experiment = client.get_experiment_by_name(EXPERIMENT_NAME)
if experiment is None:
    raise ValueError(f'Experiment not found: {EXPERIMENT_NAME}. Complete HW02 first.')

runs = mlflow.search_runs(
    experiment_ids=[experiment.experiment_id],
    filter_string="tags.leakage_status = 'clean' and tags.selected_for_serving = 'true'",
    order_by=['metrics.f1 DESC'],
)

if runs.empty:
    raise ValueError(
        'No run tagged selected_for_serving=true found. '
        'Go to MLflow UI, find your best clean run, and add the tag.'
    )

BEST_RUN_ID = runs.iloc[0]['run_id']
MODEL_URI = f'runs:/{BEST_RUN_ID}/model'

print('Best run ID  :', BEST_RUN_ID)
print('Model URI    :', MODEL_URI)
print('Run name     :', runs.iloc[0].get('tags.mlflow.runName'))
print('F1 score     :', runs.iloc[0].get('metrics.f1'))


Best run ID  : a37a223cfd294ac2a27516b90d5a795c
Model URI    : runs:/a37a223cfd294ac2a27516b90d5a795c/model
Run name     : v5_random_forest
F1 score     : 0.9863013698630136


In [3]:
model = mlflow.sklearn.load_model(MODEL_URI)
print('Model type:', type(model))
print('Model steps:', list(model.named_steps.keys()) if hasattr(model, 'named_steps') else 'not a pipeline')

Model type: <class 'sklearn.pipeline.Pipeline'>
Model steps: ['preprocessor', 'classifier']


### 1.2 Load your HW01 feature dataset

You will use a small sample from your HW01 feature parquet file to verify reproducibility.

In [5]:
FEATURE_COLS = [
    'room_type', 'property_type', 'neighbourhood_name',
    'accommodates', 'bedrooms', 'beds', 'bathrooms', 'listing_price',
    'minimum_nights', 'maximum_nights', 'instant_bookable', 'is_superhost',
    'host_listing_count', 'total_reviews_before_cutoff', 'unique_reviewers_before_cutoff',
    'avg_comment_len_before_cutoff', 'max_comment_len_before_cutoff',
    'days_since_last_review', 'available_days_last_90d', 'available_rate_last_90d',
    'avg_minimum_nights_calendar_last_90d', 'avg_maximum_nights_calendar_last_90d',
    'available_days_last_30d', 'available_rate_last_30d',
    'avg_minimum_nights_calendar_last_30d', 'avg_maximum_nights_calendar_last_30d',
]
TARGET_COL = 'high_demand_proxy'

# Load your HW01 parquet file
# Adjust the path if needed
parquet_path = list(Path('data/features').glob('*.parquet'))
if not parquet_path:
    raise FileNotFoundError('HW01 feature parquet not found. Run HW01 ETL first.')

df = pd.read_parquet(parquet_path[0])
print('Dataset shape:', df.shape)
df[FEATURE_COLS + [TARGET_COL]].head(3)


Dataset shape: (10480, 33)


,room_type,property_type,neighbourhood_name,accommodates,bedrooms,beds,bathrooms,listing_price,minimum_nights,maximum_nights,...,days_since_last_review,available_days_last_90d,available_rate_last_90d,avg_minimum_nights_calendar_last_90d,avg_maximum_nights_calendar_last_90d,available_days_last_30d,available_rate_last_30d,avg_minimum_nights_calendar_last_30d,avg_maximum_nights_calendar_last_30d,high_demand_proxy
0,Entire home/apt,Entire rental unit,Buitenveldert - Zuidas,2,1.0,1.0,1.0,146.0,2,30,...,365.0,91,1.0,2.0,30.0,30,1.0,2.0,30.0,0
1,Entire home/apt,Entire rental unit,Zuid,2,1.0,NaN,1.5,NaN,5,25,...,668.0,0,0.0,5.0,25.0,0,0.0,5.0,25.0,1
2,Entire home/apt,Entire condo,Centrum-Oost,2,1.0,NaN,1.0,NaN,2,7,...,347.0,0,0.0,2.0,7.0,0,0.0,2.0,7.0,1


### 1.3 Reproducibility check

**TODO 1.3**

Take a sample of 50 rows from the dataset.

Run `model.predict()` on those rows **twice** and verify the results are identical.

Then compare the predictions against the `high_demand_proxy` column and print:
- how many predictions match the training label
- the accuracy on this sample

If both runs produce identical output, print `Reproducibility check passed.`
If they differ, raise a `ValueError`.

In [7]:
# TODO 1.3
sample = df[FEATURE_COLS + [TARGET_COL]].sample(50, random_state=42)
X_sample = sample[FEATURE_COLS]
y_sample = sample[TARGET_COL]

pred_run_1 = model.predict(X_sample)
pred_run_2 = model.predict(X_sample)

if not np.array_equal(pred_run_1, pred_run_2):
    raise ValueError('Predictions differ between two identical predict() calls.')

matches = int((pred_run_1 == y_sample.to_numpy()).sum())
accuracy = matches / len(y_sample)

print(f'Matching predictions: {matches}/{len(y_sample)}')
print(f'Accuracy on sample: {accuracy:.4f}')
print('Reproducibility check passed.')


Matching predictions: 49/50
Accuracy on sample: 0.9800
Reproducibility check passed.


---
## Part 2 — FastAPI Service

A REST API is the standard way to expose an ML model to other systems.

You will build a FastAPI app with two prediction endpoints:
- `POST /predict` — single listing prediction
- `POST /predict/batch` — multiple listings in one request

Then you will measure how much faster batch is compared to calling single predict in a loop.

### 2.1 Input and output schemas

In [ ]:
# TODO 2.1 - implemented in src/airbnb_serving/schema.py

from airbnb_serving.schema import ListingFeatures, PredictionResponse, FEATURE_COLS

assert 'BaseModel' in (PROJECT / 'src/airbnb_serving/schema.py').read_text()
print('schema.py OK —', len(FEATURE_COLS), 'feature fields')


schema.py OK — 26 feature fields


### 2.2 Prediction logic

In [ ]:
# TODO 2.2 - implemented in src/airbnb_serving/predictor.py

from airbnb_serving.predictor import predict_single, predict_batch

content = (PROJECT / 'src/airbnb_serving/predictor.py').read_text()
assert 'predict_single' in content and 'predict_batch' in content
print('predictor.py OK')


predictor.py OK


### 2.3 FastAPI app

In [ ]:
# TODO 2.3 - implemented in src/airbnb_serving/app.py

content = (PROJECT / 'src/airbnb_serving/app.py').read_text()
for endpoint in ['/health', '/predict', '/predict/batch']:
    assert endpoint in content, f'Missing endpoint: {endpoint}'
assert 'lifespan' in content
print('app.py OK')


app.py OK


### 2.4 Package metadata

In [ ]:
# TODO 2.4 - implemented in pyproject.toml and requirements.txt

assert (PROJECT / 'pyproject.toml').exists()
assert (PROJECT / 'requirements.txt').exists()
print('pyproject.toml OK')
print('requirements.txt OK')


pyproject.toml OK
requirements.txt OK


### 2.5 Local install and smoke test

Install the package and manually start the server in a terminal before running the test cell below.

```bash
pip install -e .

MODEL_RUN_ID=<your_run_id> \
MLFLOW_TRACKING_URI=http://185.50.38.163:33014 \
MLFLOW_TRACKING_USERNAME=<user> \
MLFLOW_TRACKING_PASSWORD=<pass> \
uvicorn airbnb_serving.app:app --host 0.0.0.0 --port 8000
```

In [ ]:
import subprocess
import sys

subprocess.check_call(
    [sys.executable, '-m', 'pip', 'install', '-q', '-e', '.'],
    cwd=PROJECT,
)

0

In [ ]:
import subprocess, time, signal, os

server_proc = subprocess.Popen(
    ['uvicorn', 'airbnb_serving.app:app', '--host', '0.0.0.0', '--port', '12345'],
    env={**os.environ, 'MODEL_RUN_ID': BEST_RUN_ID},
)
time.sleep(15)  # wait for model to load from MLflow
print('Server started, PID:', server_proc.pid)

INFO:     Started server process [114929]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:12345 (Press CTRL+C to quit)


Server started, PID: 114929


INFO:     127.0.0.1:51124 - "GET / HTTP/1.1" 404 Not Found
INFO:     127.0.0.1:51128 - "GET /favicon.ico HTTP/1.1" 404 Not Found
INFO:     127.0.0.1:51124 - "GET /docs HTTP/1.1" 200 OK
INFO:     127.0.0.1:51124 - "GET /openapi.json HTTP/1.1" 200 OK


In [15]:
import requests

BASE_URL = 'http://localhost:12345'

health = requests.get(f'{BASE_URL}/health')
assert health.status_code == 200, f'Health check failed: {health.text}'
print('Health:', health.json())

sample_payload = {
    'room_type': 'Entire home/apt',
    'property_type': 'Entire rental unit',
    'neighbourhood_name': 'Centrum-West',
    'accommodates': 2,
    'bedrooms': 1.0,
    'beds': 1.0,
    'bathrooms': 1.0,
    'listing_price': 150.0,
    'minimum_nights': 2,
    'maximum_nights': 365,
    'instant_bookable': True,
    'is_superhost': False,
    'host_listing_count': 1,
    'total_reviews_before_cutoff': 10.0,
    'unique_reviewers_before_cutoff': 9.0,
    'avg_comment_len_before_cutoff': 120.0,
    'max_comment_len_before_cutoff': 300.0,
    'days_since_last_review': 30.0,
    'available_days_last_90d': 45,
    'available_rate_last_90d': 0.5,
    'avg_minimum_nights_calendar_last_90d': 2.0,
    'avg_maximum_nights_calendar_last_90d': 365.0,
    'available_days_last_30d': 15,
    'available_rate_last_30d': 0.5,
    'avg_minimum_nights_calendar_last_30d': 2.0,
    'avg_maximum_nights_calendar_last_30d': 365.0,
}

resp = requests.post(f'{BASE_URL}/predict', json=sample_payload)
assert resp.status_code == 200, f'Single predict failed: {resp.text}'
print('Single predict:', resp.json())

batch_resp = requests.post(f'{BASE_URL}/predict/batch', json=[sample_payload, sample_payload])
assert batch_resp.status_code == 200, f'Batch predict failed: {batch_resp.text}'
print('Batch predict count:', len(batch_resp.json()))

print('Local smoke test passed.')


INFO:     127.0.0.1:36768 - "GET /health HTTP/1.1" 200 OK
Health: {'status': 'ok', 'model_run_id': 'a37a223cfd294ac2a27516b90d5a795c'}
INFO:     127.0.0.1:36780 - "POST /predict HTTP/1.1" 200 OK
Single predict: {'listing_id': None, 'prediction': 0, 'probability_high_demand': 0.1579640105184853, 'model_run_id': 'a37a223cfd294ac2a27516b90d5a795c'}
INFO:     127.0.0.1:36782 - "POST /predict/batch HTTP/1.1" 200 OK
Batch predict count: 2
Local smoke test passed.


### 2.6 Batch vs single benchmark

**TODO 2.6**

Take 100 rows from your feature dataset.

Measure:
1. Time to call `POST /predict` 100 times in a loop (single)
2. Time to call `POST /predict/batch` once with all 100 rows (batch)

Print a comparison table with total time and time per prediction for each approach.

Then answer: why is batch faster? Write your answer as a comment in the cell.

In [16]:
# TODO 2.6
import math

def clean_for_json(row: dict) -> dict:
    return {
        k: None if isinstance(v, float) and math.isnan(v) else v
        for k, v in row.items()
    }

BENCHMARK_SIZE = 100
benchmark_rows = [
    clean_for_json(row)
    for row in df[FEATURE_COLS].head(BENCHMARK_SIZE).to_dict(orient='records')
]

start = time.perf_counter()
for row in benchmark_rows:
    requests.post(f'{BASE_URL}/predict', json=row)
single_total = time.perf_counter() - start

start = time.perf_counter()
requests.post(f'{BASE_URL}/predict/batch', json=benchmark_rows)
batch_total = time.perf_counter() - start

single_per_ms = single_total / BENCHMARK_SIZE * 1000
batch_per_ms = batch_total / BENCHMARK_SIZE * 1000
speedup = single_total / batch_total

print(f"{'Method':<14} | {'Total (s)':>9} | {'Per prediction (ms)':>20}")
print(f"{'single loop':<14} | {single_total:>9.2f} | {single_per_ms:>20.1f}")
print(f"{'batch':<14} | {batch_total:>9.2f} | {batch_per_ms:>20.1f}")
print(f'Speedup: {speedup:.1f}x')

# Why is batch faster?
# Batch sends one HTTP request and runs model.predict once on all rows.
# The single loop pays HTTP + JSON overhead 100 times and calls the model separately each time.


INFO:     127.0.0.1:36788 - "POST /predict HTTP/1.1" 200 OK
INFO:     127.0.0.1:36802 - "POST /predict HTTP/1.1" 200 OK
INFO:     127.0.0.1:36812 - "POST /predict HTTP/1.1" 200 OK
INFO:     127.0.0.1:36814 - "POST /predict HTTP/1.1" 200 OK
INFO:     127.0.0.1:36818 - "POST /predict HTTP/1.1" 200 OK
INFO:     127.0.0.1:36822 - "POST /predict HTTP/1.1" 200 OK
INFO:     127.0.0.1:36832 - "POST /predict HTTP/1.1" 200 OK
INFO:     127.0.0.1:36844 - "POST /predict HTTP/1.1" 200 OK
INFO:     127.0.0.1:36852 - "POST /predict HTTP/1.1" 200 OK
INFO:     127.0.0.1:36858 - "POST /predict HTTP/1.1" 200 OK
INFO:     127.0.0.1:36860 - "POST /predict HTTP/1.1" 200 OK
INFO:     127.0.0.1:36868 - "POST /predict HTTP/1.1" 200 OK
INFO:     127.0.0.1:36870 - "POST /predict HTTP/1.1" 200 OK
INFO:     127.0.0.1:36878 - "POST /predict HTTP/1.1" 200 OK
INFO:     127.0.0.1:36884 - "POST /predict HTTP/1.1" 200 OK
INFO:     127.0.0.1:36888 - "POST /predict HTTP/1.1" 200 OK
INFO:     127.0.0.1:36892 - "POST /predi

In [17]:
server_proc.terminate()
print('Server stopped.')

Server stopped.


INFO:     Shutting down
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.
INFO:     Finished server process [114929]


---
## Part 3 — Docker

You will build two Docker images and compare their sizes.

This teaches you that image size is not free — it affects pull time, storage cost, and attack surface.

### 3.1 Naive Dockerfile

In [ ]:
# TODO 3.1 - implemented in Dockerfile.naive

assert (PROJECT / 'Dockerfile.naive').exists()
content = (PROJECT / 'Dockerfile.naive').read_text()
assert 'FROM python:3.11' in content
assert 'COPY . .' in content
print('Dockerfile.naive OK')


Dockerfile.naive OK


### 3.2 Optimized Dockerfile

A multi-stage build separates the build environment from the runtime environment.

Stage 1 (builder): install everything, build the package.
Stage 2 (runtime): copy only what is needed to run, nothing else.

In [ ]:
# TODO 3.2 - implemented in Dockerfile and .dockerignore

assert (PROJECT / 'Dockerfile').exists()
assert (PROJECT / '.dockerignore').exists()
dockerfile = (PROJECT / 'Dockerfile').read_text()
assert 'AS builder' in dockerfile
assert 'COPY --from=builder' in dockerfile
assert '.env' in (PROJECT / '.dockerignore').read_text()
print('Dockerfile OK')
print('.dockerignore OK')


Dockerfile OK
.dockerignore OK


### 3.3 Build and compare image sizes

In [ ]:
import subprocess

build_dir = PROJECT  
tags = ['naive', 'optimized']
existing = subprocess.run(
    ['docker', 'images', 'qbc12-airbnb-serving', '--format', '{{.Tag}}'],
    capture_output=True, text=True, cwd=build_dir,
).stdout.split()

if all(tag in existing for tag in tags):
    print('Both images already built — skipping rebuild.')
    print('To force a rebuild, run in terminal:')
    print('  docker build -f Dockerfile.naive -t qbc12-airbnb-serving:naive .')
    print('  docker build -f Dockerfile -t qbc12-airbnb-serving:optimized .')
else:
    builds = [
        (['docker', 'build', '-f', 'Dockerfile.naive', '-t', 'qbc12-airbnb-serving:naive', '.'], 'naive'),
        (['docker', 'build', '-f', 'Dockerfile', '-t', 'qbc12-airbnb-serving:optimized', '.'], 'optimized'),
    ]
    for cmd, name in builds:
        if name in existing:
            print(f'Skipping {name} — image already exists.')
            continue
        print(f'\n=== Building {name} image (this can take 10-20 min the first time) ===')
        result = subprocess.run(cmd, cwd=build_dir)
        if result.returncode != 0:
            raise RuntimeError(f'Docker build failed for {name} (exit code {result.returncode})')
        print(f'=== {name} image built successfully ===')

print('\n=== Installed images ===')
subprocess.run(['docker', 'images', 'qbc12-airbnb-serving'], cwd=build_dir)
print('\nDocker build step complete.')

Both images already built — skipping rebuild.
To force a rebuild, run in terminal:
  docker build -f Dockerfile.naive -t qbc12-airbnb-serving:naive .
  docker build -f Dockerfile -t qbc12-airbnb-serving:optimized .

=== Installed images ===
IMAGE                            ID             DISK USAGE   CONTENT SIZE   EXTRA
qbc12-airbnb-serving:naive       f164a2a1f563       1.91GB             0B        
qbc12-airbnb-serving:optimized   58ee0c7d7608        923MB             0B        

Docker build step complete.


In [ ]:
# TODO 3.3  - implemented in reports/docker_size_report.md

import subprocess, json

result = subprocess.run(
    ['docker', 'images', '--format', '{{json .}}'],
    capture_output=True, text=True
)

images = [json.loads(line) for line in result.stdout.strip().split('\n') if line]
serving_images = [
    img for img in images
    if img.get('Repository') == 'qbc12-airbnb-serving'
]

size_df = pd.DataFrame(serving_images)[['Repository', 'Tag', 'Size']]
print(size_df.to_string(index=False))

markdown_table = [
    '| Repository | Tag | Size |',
    '|---|---|---|',
]
for row in size_df.itertuples(index=False):
    markdown_table.append(f'| {row.Repository} | {row.Tag} | {row.Size} |')

report_lines = [
    '# HW03 Docker Image Size Report', '',
    *markdown_table, '',
    '## Analysis',
    (
        'The naive image (1.91GB) is about twice the size of the optimized image (923MB). '
        'It uses the full `python:3.11` base image and `COPY . .`, so the build context '
        'includes notebooks, reports, and other files that are not needed to run the API.'
    ),
    (
        'The optimized image uses `python:3.11-slim`, a multi-stage build, and a '
        '`.dockerignore` file. Dependencies are installed in a builder stage; only the '
        'installed packages and `src/` are copied into the final runtime image.'
    ),
    (
        'For production I would deploy the optimized image: it is smaller to pull and store, '
        'has a smaller attack surface, and runs the same FastAPI service with the model '
        'loaded from MLflow at startup.'
    ),
]
Path('reports/docker_size_report.md').write_text('\n'.join(report_lines) + '\n')
print('\nReport saved to reports/docker_size_report.md')


          Repository       Tag   Size
qbc12-airbnb-serving     naive 1.91GB
qbc12-airbnb-serving optimized  923MB

Report saved to reports/docker_size_report.md


### 3.4 Docker Compose

In [ ]:
# TODO 3.4 - implemented in docker-compose.yml, .env.example, and .gitignore

assert (PROJECT / 'docker-compose.yml').exists()
assert (PROJECT / '.env.example').exists()
assert '.env' in (PROJECT / '.gitignore').read_text()
compose = (PROJECT / 'docker-compose.yml').read_text()
assert 'airbnb-serving' in compose
assert 'qbc12-airbnb-serving:optimized' in compose
assert 'env_file' in compose
print('docker-compose.yml OK')
print('.env.example OK')


docker-compose.yml OK
.env.example OK


In [28]:
# Docker Compose smoke test
!docker compose up -d

import time, requests
time.sleep(8)  # wait for model to load from MLflow

health = requests.get('http://localhost:8000/health')
assert health.status_code == 200, f'Failed: {health.text}'
print('Docker Compose health check passed:', health.json())

!docker compose down

[+] Running 1/1
 ✔ Container hw03_model_serving-airbnb-serving-1  Running                  0.0s 
Docker Compose health check passed: {'status': 'ok', 'model_run_id': 'a37a223cfd294ac2a27516b90d5a795c'}
[+] Running 0/1
 ⠋ Container hw03_model_serving-airbnb-serving-1  Stopping                 0.1s 
[+] Running 0/1
 ⠙ Container hw03_model_serving-airbnb-serving-1  Stopping                 0.2s 
[+] Running 0/1
 ⠹ Container hw03_model_serving-airbnb-serving-1  Stopping                 0.3s 
[+] Running 0/1
 ⠸ Container hw03_model_serving-airbnb-serving-1  Stopping                 0.4s 
[+] Running 0/1
 ⠼ Container hw03_model_serving-airbnb-serving-1  Stopping                 0.5s 
[+] Running 0/1
 ⠴ Container hw03_model_serving-airbnb-serving-1  Stopping                 0.6s 
[+] Running 0/1
 ⠦ Container hw03_model_serving-airbnb-serving-1  Stopping                 0.7s 
[+] Running 0/1
 ⠧ Container hw03_model_serving-airbnb-serving-1  Stopping                 0.8s 
[+] Running 0/1
 ⠇ Con

---
## Part 4 — Kubernetes Manifests

Kubernetes is the standard way to run containers in production at scale.

You do not need a real cluster for this homework. The deliverable is correct YAML files that a cluster could apply.

Key concepts you will use:

| Concept | What it does |
|---|---|
| **Pod** | Runs your container |
| **Deployment** | Manages multiple identical Pods, handles restarts |
| **Service** | Stable network endpoint that routes traffic to Pods |
| **Secret** | Stores sensitive values like passwords, not plaintext in YAML |
| **readinessProbe** | Tells Kubernetes when a Pod is ready to receive traffic |
| **resource limits** | Prevents one Pod from consuming all server memory |

### 4.1 Deployment

In [ ]:
# TODO 4.1 - implemented in k8s/deployment.yaml

deployment = (PROJECT / 'k8s/deployment.yaml').read_text()
assert (PROJECT / 'k8s/deployment.yaml').exists()
assert 'kind: Deployment' in deployment
assert 'name: airbnb-serving' in deployment
assert 'replicas: 2' in deployment
assert 'qbc12-airbnb-serving:optimized' in deployment
assert 'airbnb-serving-secret' in deployment
assert 'MODEL_RUN_ID' in deployment
assert 'readinessProbe' in deployment
assert 'initialDelaySeconds: 15' in deployment
print('deployment.yaml OK')

deployment.yaml OK


### 4.2 Service

In [ ]:
# TODO 4.2 - implemented in k8s/deployment.yaml

service = (PROJECT / 'k8s/service.yaml').read_text()
assert (PROJECT / 'k8s/service.yaml').exists()
assert 'kind: Service' in service
assert 'name: airbnb-serving' in service
assert 'type: ClusterIP' in service
assert 'targetPort: 8000' in service
assert 'app: airbnb-serving' in service
print('service.yaml OK')

service.yaml OK


### 4.3 Conceptual questions

Answer these in the markdown cell below. One or two sentences each is enough.

**TODO 4.3 — Answer here:**

**Q1.** We set `replicas: 2` instead of 1. What happens to traffic if one Pod crashes while replicas is 1 vs 2?

A: With `replicas: 1`, the only Pod handles all traffic. If it crashes, the Service has no healthy backend until Kubernetes restarts it, so requests fail or time out during downtime. With `replicas: 2`, traffic is spread across two Pods. If one crashes, the other keeps serving requests while Kubernetes replaces the failed Pod, so the API stays available.

**Q2.** The `readinessProbe` has `initialDelaySeconds: 15`. Why do we need a delay specifically for this service?

A: On startup the app downloads and loads the sklearn model from MLflow before it can answer `/health` successfully. Without the delay, Kubernetes would probe too early, mark the Pod as not ready, and withhold traffic even though the container is still initializing normally.

**Q3.** Why do we store credentials in a Kubernetes Secret instead of writing them directly in `deployment.yaml`?

A: Secrets keep passwords and tokens out of version control and plain YAML files that many people can read. They can be managed separately, rotated without changing the Deployment manifest, and access can be restricted with Kubernetes RBAC.

**Q4.** What is the difference between `ClusterIP` and `LoadBalancer` service types? When would you use each?

A: `ClusterIP` exposes the service only inside the cluster on an internal virtual IP. Use it when other Pods or in-cluster clients call the API. `LoadBalancer` provisions an external IP or cloud load balancer so traffic from outside the cluster can reach the service. Use it when you need public internet access to the API.

---
## Final Proof

If this cell fails, HW03 is not complete.

In [31]:
required_files = [
    'src/airbnb_serving/__init__.py',
    'src/airbnb_serving/schema.py',
    'src/airbnb_serving/predictor.py',
    'src/airbnb_serving/app.py',
    'pyproject.toml',
    'requirements.txt',
    'Dockerfile',
    'Dockerfile.naive',
    'docker-compose.yml',
    '.env.example',
    '.dockerignore',
    'k8s/deployment.yaml',
    'k8s/service.yaml',
    'reports/docker_size_report.md',
]

missing = [f for f in required_files if not (PROJECT / f).exists()]
assert not missing, f'Missing files:\n' + '\n'.join(missing)

# Check .env is gitignored
gitignore = (PROJECT / '.gitignore').read_text() if (PROJECT / '.gitignore').exists() else ''
assert '.env' in gitignore, '.env must be in .gitignore'

# Check Dockerfile does not copy .env
for df_name in ['Dockerfile', 'Dockerfile.naive']:
    content = (PROJECT / df_name).read_text()
    assert 'COPY .env' not in content, f'Do not copy .env in {df_name}'

# Check schema.py has actual content
schema_content = (PROJECT / 'src/airbnb_serving/schema.py').read_text()
assert 'BaseModel' in schema_content, 'schema.py must define Pydantic models'

# Check app.py has endpoints
app_content = (PROJECT / 'src/airbnb_serving/app.py').read_text()
assert '/health' in app_content, 'app.py must have /health endpoint'
assert '/predict' in app_content, 'app.py must have /predict endpoint'
assert 'batch' in app_content, 'app.py must have /predict/batch endpoint'

print('All required files present.')
print('No credential leaks detected.')
print('HW03 proof passed.')

All required files present.
No credential leaks detected.
HW03 proof passed.


## Screenshots required

Add these to the `screenshots/` folder before submitting:

- `screenshots/health_endpoint.png` — GET /health response
- `screenshots/predict_endpoint.png` — POST /predict response
- `screenshots/batch_endpoint.png` — POST /predict/batch response
- `screenshots/fastapi_docs.png` — auto-generated docs at /docs
- `screenshots/docker_image_sizes.png` — output of `docker images` showing both image sizes

## Commit

```bash
git add .
git commit -m "HW03 model serving and deployment"
git push
```